Gerar Exceis

In [8]:
# ============================================================
# DEGRADAÇÃO TEMPORAL REALISTA DE CORROSÃO
# - Só cresce anomalias com coluna C == "corrosion"
# - "damage" e "water" mantêm-se inalteradas
# - Gera 20 ficheiros Excel (de 6 em 6 meses, durante 10 anos)
# - Crescimento desacelerado ao longo do tempo
# ============================================================

import re
import math
from pathlib import Path

import numpy as np
import cv2
from openpyxl import load_workbook
from PIL import Image

# ============================================================
# CONFIG
# ============================================================

# Pasta onde está o excel original
FICHEIRO_EXCEL = "metricas_anomalias_faces_malha.xlsx"

# Pasta das imagens base para obter largura/altura corretas
PASTA_IMAGENS = 'image'

# Pasta onde serão guardados os 20 Excels
PASTA_SAIDA_EXCELS = 'excel_result/degradacao_temporal_excels'

# ------------------------------------------------------------
# PARÂMETROS TEMPORAIS
# ------------------------------------------------------------
NUM_PASSOS = 20              # 20 excels
MESES_POR_PASSO = 6          # de 6 em 6 meses

# ------------------------------------------------------------
# PARÂMETROS DE DEGRADAÇÃO
# ------------------------------------------------------------
# Kernel da dilatação
KERNEL_SIZE = 3

# "Força" inicial de degradação por passo.
# Não precisa de ser inteira. O acumulador trata das frações.
BASE_STEP_STRENGTH = 1.8

# Quanto maior, mais rapidamente a degradação abranda.
# Valores típicos: 1.5 a 4.0
DECAY_RATE = 2.8

# Mínimo de crescimento residual relativo no último passo
# evita que a degradação "morra completamente"
# Ex.: 0.15 = no fim ainda há 15% da força inicial
MIN_RELATIVE_GROWTH = 0.15

# Se True, aplica fechamento morfológico leve após dilatação
# para tapar pequenas falhas/aberturas e deixar o dano mais coeso
APPLY_MORPH_CLOSE = True
MORPH_CLOSE_KERNEL_SIZE = 3

AREA_DECIMALS = 6
EXTENSOES_VALIDAS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def parse_polygon_string(poly_str):
    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()
    if not poly_str:
        return []

    matches = re.findall(r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]", poly_str)
    return [(float(x), float(y)) for x, y in matches]


def polygon_to_string(points):
    return "//".join(f"[{int(round(x))},{int(round(y))}]" for x, y in points)


def polygon_area_pixels(points):
    if len(points) < 3:
        return 0.0

    pts = np.array(points, dtype=float)
    x = pts[:, 0]
    y = pts[:, 1]
    area = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
    return float(area)


def safe_float(v, default=0.0):
    try:
        return float(v)
    except:
        return default


def normalize_anomaly_type(v):
    if v is None:
        return ""
    return str(v).strip().lower()


def index_images_by_name(root_folder):
    root = Path(root_folder)
    image_map = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[p.name.lower()] = p
    return image_map


def get_image_size_cached(image_name, image_index, size_cache):
    if image_name in size_cache:
        return size_cache[image_name]

    img_path = image_index.get(image_name)
    if img_path is None:
        return None

    with Image.open(img_path) as img_ref:
        size_cache[image_name] = img_ref.size  # (width, height)

    return size_cache[image_name]


def polygon_to_mask(points, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)

    pts = np.array(
        [[
            [
                min(max(int(round(x)), 0), width - 1),
                min(max(int(round(y)), 0), height - 1)
            ]
            for x, y in points
        ]],
        dtype=np.int32
    )

    if pts.shape[1] >= 3:
        cv2.fillPoly(mask, pts, 255)

    return mask


def dilate_mask(mask, kernel_size=3, iterations=1):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=iterations)
    return dilated


def close_mask(mask, kernel_size=3):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)


def mask_to_polygon(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return []

    contour = max(contours, key=cv2.contourArea)

    epsilon = 0.002 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)

    points = [(float(pt[0][0]), float(pt[0][1])) for pt in approx]
    return points


def get_step_strength(step_idx, total_steps, base_strength, decay_rate, min_relative_growth):
    """
    step_idx: 1..total_steps
    Retorna força relativa decrescente ao longo do tempo.
    No início ~1.0
    No fim ~min_relative_growth
    """
    if total_steps <= 1:
        return base_strength

    s = (step_idx - 1) / (total_steps - 1)  # 0..1

    # exponencial decrescente normalizada
    relative = math.exp(-decay_rate * s)

    # remap para nunca cair abaixo de min_relative_growth
    relative = min_relative_growth + (1.0 - min_relative_growth) * relative

    return base_strength * relative


def build_output_filename(base_name, step_idx, months_elapsed):
    stem = Path(base_name).stem
    suffix = Path(base_name).suffix
    years = months_elapsed // 12
    rem_months = months_elapsed % 12
    return f"{stem}_degradacao_t{step_idx:02d}_{years:02d}a_{rem_months:02d}m{suffix}"


# ============================================================
# PREPARAÇÃO
# ============================================================

excel_path = FICHEIRO_EXCEL
output_dir = Path(PASTA_SAIDA_EXCELS)
output_dir.mkdir(parents=True, exist_ok=True)


image_index = index_images_by_name(PASTA_IMAGENS)
if not image_index:
    raise RuntimeError(f"Não foram encontradas imagens em: {PASTA_IMAGENS}")

print(f"[INFO] Excel base: {excel_path}")
print(f"[INFO] Pasta imagens: {PASTA_IMAGENS}")
print(f"[INFO] Pasta saída excels: {output_dir}")
print(f"[INFO] Imagens indexadas: {len(image_index)}")

# cache de tamanhos de imagem
image_size_cache = {}

# workbook que vai sendo degradado cumulativamente
wb = load_workbook(excel_path)

# acumulador por anomalia para permitir forças fracionárias
# chave única por linha: (sheet_name, row)
growth_budget = {}

# ============================================================
# CICLO TEMPORAL
# ============================================================

for step in range(1, NUM_PASSOS + 1):
    months_elapsed = step * MESES_POR_PASSO

    # força deste semestre
    step_strength = get_step_strength(
        step_idx=step,
        total_steps=NUM_PASSOS,
        base_strength=BASE_STEP_STRENGTH,
        decay_rate=DECAY_RATE,
        min_relative_growth=MIN_RELATIVE_GROWTH
    )

    print("\n" + "=" * 68)
    print(f"[PASSO {step:02d}/{NUM_PASSOS}]  Tempo = {months_elapsed} meses")
    print(f"[INFO] Força teórica deste passo = {step_strength:.4f}")
    print("=" * 68)

    total_corrosion_updated = 0
    total_non_corrosion_kept = 0
    total_skipped = 0

    for ws in wb.worksheets:
        print(f"\n--- Folha: {ws.title} ---")

        corrosion_updated_sheet = 0
        non_corrosion_kept_sheet = 0
        skipped_sheet = 0

        for row in range(2, ws.max_row + 1):
            anomaly_type = normalize_anomaly_type(ws[f"C{row}"].value)
            polygon_str = ws[f"E{row}"].value
            area_cm2_old = ws[f"F{row}"].value
            image_path_excel = ws[f"G{row}"].value

            # Se não houver geometria ou imagem associada, ignora
            if not polygon_str or not image_path_excel:
                skipped_sheet += 1
                total_skipped += 1
                continue

            points = parse_polygon_string(polygon_str)
            if len(points) < 3:
                skipped_sheet += 1
                total_skipped += 1
                continue

            # Só corrosão cresce
            if anomaly_type != "corrosion":
                non_corrosion_kept_sheet += 1
                total_non_corrosion_kept += 1
                continue

            image_name = Path(str(image_path_excel)).name.lower()
            size = get_image_size_cached(image_name, image_index, image_size_cache)

            if size is None:
                skipped_sheet += 1
                total_skipped += 1
                continue

            width, height = size

            old_pixel_area = polygon_area_pixels(points)
            old_cm2 = safe_float(area_cm2_old, default=0.0)

            if old_pixel_area <= 0:
                skipped_sheet += 1
                total_skipped += 1
                continue

            row_key = (ws.title, row)
            prev_budget = growth_budget.get(row_key, 0.0)
            new_budget = prev_budget + step_strength

            # quantas iterações inteiras aplicar neste passo
            iterations_to_apply = int(math.floor(new_budget))
            growth_budget[row_key] = new_budget - iterations_to_apply

            # Se neste semestre ainda não atingiu 1 iteração,
            # mantém como está e passa orçamento sobrante ao próximo passo
            if iterations_to_apply <= 0:
                corrosion_updated_sheet += 1
                total_corrosion_updated += 1
                continue

            # 1) polígono -> máscara
            mask = polygon_to_mask(points, width, height)

            # 2) dilatar máscara
            grown_mask = dilate_mask(
                mask,
                kernel_size=KERNEL_SIZE,
                iterations=iterations_to_apply
            )

            # 3) opcional: fechar pequenas falhas
            if APPLY_MORPH_CLOSE:
                grown_mask = close_mask(grown_mask, kernel_size=MORPH_CLOSE_KERNEL_SIZE)

            # 4) máscara -> novo polígono
            new_points = mask_to_polygon(grown_mask)

            if len(new_points) < 3:
                skipped_sheet += 1
                total_skipped += 1
                continue

            new_pixel_area = polygon_area_pixels(new_points)

            # mesma conversão implícita pixel² -> cm² do registo original
            if old_cm2 > 0:
                cm2_per_pixel2 = old_cm2 / old_pixel_area
                new_cm2 = new_pixel_area * cm2_per_pixel2
            else:
                new_cm2 = 0.0

            # atualizar Excel em memória
            ws[f"E{row}"] = polygon_to_string(new_points)
            ws[f"F{row}"] = round(new_cm2, AREA_DECIMALS)

            corrosion_updated_sheet += 1
            total_corrosion_updated += 1

        print(f"Corrosion processadas:      {corrosion_updated_sheet}")
        print(f"Damage/Water mantidas:      {non_corrosion_kept_sheet}")
        print(f"Ignoradas por erro/falta:   {skipped_sheet}")

    # guardar snapshot deste instante temporal
    output_name = build_output_filename(
        base_name=FICHEIRO_EXCEL,
        step_idx=step,
        months_elapsed=months_elapsed
    )
    output_path = output_dir / output_name
    wb.save(output_path)

    print("\n[SNAPSHOT GUARDADO]")
    print(f"Ficheiro: {output_path}")
    print(f"Corrosion processadas: {total_corrosion_updated}")
    print(f"Damage/Water mantidas: {total_non_corrosion_kept}")
    print(f"Ignoradas:             {total_skipped}")

print("\n" + "=" * 68)
print("PROCESSO COMPLETO")
print("=" * 68)
print(f"Excel base: {excel_path}")
print(f"Excels gerados: {NUM_PASSOS}")
print(f"Pasta de saída: {output_dir}")

[INFO] Excel base: metricas_anomalias_faces_malha.xlsx
[INFO] Pasta imagens: image
[INFO] Pasta saída excels: excel_result\degradacao_temporal_excels
[INFO] Imagens indexadas: 831

[PASSO 01/20]  Tempo = 6 meses
[INFO] Força teórica deste passo = 1.8000

--- Folha: Anomalies00 ---
Corrosion processadas:      0
Damage/Water mantidas:      662
Ignoradas por erro/falta:   2191

--- Folha: corrosion ---
Corrosion processadas:      0
Damage/Water mantidas:      0
Ignoradas por erro/falta:   2192

--- Folha: damage ---
Corrosion processadas:      0
Damage/Water mantidas:      651
Ignoradas por erro/falta:   1

--- Folha: water ---
Corrosion processadas:      0
Damage/Water mantidas:      11
Ignoradas por erro/falta:   1

[SNAPSHOT GUARDADO]
Ficheiro: excel_result\degradacao_temporal_excels\metricas_anomalias_faces_malha_degradacao_t01_00a_06m.xlsx
Corrosion processadas: 0
Damage/Water mantidas: 1324
Ignoradas:             4385

[PASSO 02/20]  Tempo = 12 meses
[INFO] Força teórica deste passo

Reconstruir Imagens

In [15]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
import re

from openpyxl import load_workbook
from PIL import Image, ImageDraw

# ============================================================
# CONFIG
# ============================================================

PASTA_EXCELS_EVENTOS = "excel_result/degradacao_temporal_excels"
NOVA_PASTA_IMAGENS = "image"
PASTA_SAIDA = "previews_degradacao_temporal"

SUBPASTA_OVERLAY = "overlay"
SUBPASTA_COMPARISON = "comparison"

EXCEL_GLOB = "*.xlsx"

EXTENSOES_VALIDAS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp",
    ".webp"
}

DESENHAR_CONTORNO = False
ESPESSURA_CONTORNO = 2

COMPARISON_GAP = 20
COR_GAP = (255, 255, 255)

CORES_ANOMALIA = {
    "corrosion": (255, 0, 0),
    "damage":    (0, 255, 0),
    "water":     (0, 0, 255),
}

COR_DEFAULT = (255, 255, 255)

# ============================================================
# AUXILIARES
# ============================================================

def parse_polygon_string(poly_str):
    """
    Converte:
    [x,y]//[x,y]//[x,y]
    em:
    [(x,y), (x,y), ...]
    """
    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()

    if not poly_str:
        return []

    matches = re.findall(
        r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]",
        poly_str
    )

    return [(float(x), float(y)) for x, y in matches]


def clip_points(points, width, height):
    clipped = []

    for x, y in points:
        x = min(max(int(round(x)), 0), width - 1)
        y = min(max(int(round(y)), 0), height - 1)
        clipped.append((x, y))

    return clipped


def get_color(anomaly_type):
    if anomaly_type is None:
        return COR_DEFAULT

    key = str(anomaly_type).strip().lower()
    return CORES_ANOMALIA.get(key, COR_DEFAULT)


def sanitize_stem(name):
    return re.sub(r'[\\/*?:"<>|]+', "_", name)


def index_images_by_name(root_folder):
    """
    Indexa imagens pelo nome base (sem extensão).
    Exemplo:
    Figura_1.jpg -> figura_1
    """
    root = Path(root_folder)
    image_map = {}

    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[p.stem.lower()] = p

    return image_map


def make_side_by_side(original_img, overlay_img, gap=20, gap_color=(255, 255, 255)):
    original_img = original_img.convert("RGB")
    overlay_img = overlay_img.convert("RGB")

    w1, h1 = original_img.size
    w2, h2 = overlay_img.size

    out_w = w1 + gap + w2
    out_h = max(h1, h2)

    canvas = Image.new("RGB", (out_w, out_h), gap_color)

    canvas.paste(original_img, (0, 0))
    canvas.paste(overlay_img, (w1 + gap, 0))

    return canvas


# ============================================================
# EXCEL
# ============================================================

def load_excel_grouped_by_image(excel_path):
    """
    Lê Excel e agrupa polígonos por imagem.
    """
    wb = load_workbook(excel_path, data_only=True)

    por_imagem = defaultdict(list)

    total_rows = 0
    total_valid = 0

    for ws in wb.worksheets:
        for row in range(2, ws.max_row + 1):
            total_rows += 1

            anomaly_id = ws[f"A{row}"].value
            anomaly_type = ws[f"C{row}"].value
            polygon_str = ws[f"E{row}"].value
            image_path_excel = ws[f"G{row}"].value

            if not polygon_str or not image_path_excel:
                continue

            points = parse_polygon_string(polygon_str)

            if len(points) < 3:
                continue

            image_name = PurePosixPath(
                str(image_path_excel).strip().replace("\\", "/")
            ).stem.lower()

            por_imagem[image_name].append({
                "sheet": ws.title,
                "row": row,
                "anomaly_id": anomaly_id,
                "anomaly_type": anomaly_type,
                "points": points
            })

            total_valid += 1

    return por_imagem, total_rows, total_valid


# ============================================================
# PROCESSAR EVENTO
# ============================================================

def process_one_event_excel(excel_path, image_index, output_root):
    event_name = sanitize_stem(excel_path.stem)

    event_dir = output_root / event_name
    overlay_dir = event_dir / SUBPASTA_OVERLAY
    comparison_dir = event_dir / SUBPASTA_COMPARISON

    overlay_dir.mkdir(parents=True, exist_ok=True)
    comparison_dir.mkdir(parents=True, exist_ok=True)

    por_imagem, total_rows, total_valid = load_excel_grouped_by_image(excel_path)

    print("\n" + "=" * 70)
    print(f"[EVENTO] {excel_path.name}")
    print("=" * 70)
    print(f"[INFO] Linhas totais: {total_rows}")
    print(f"[INFO] Polígonos válidos: {total_valid}")
    print(f"[INFO] Imagens com polígonos: {len(por_imagem)}")

    processadas = 0
    faltam_imagens = 0

    for image_name, registros in por_imagem.items():
        img_path = image_index.get(image_name)

        if img_path is None:
            print(f"[AVISO] Imagem não encontrada: {image_name}")
            faltam_imagens += 1
            continue

        original_img = Image.open(img_path).convert("RGB")

        out_img = original_img.copy()
        w, h = out_img.size

        draw = ImageDraw.Draw(out_img)

        for reg in registros:
            pts = clip_points(reg["points"], w, h)

            if len(pts) < 3:
                continue

            cor = get_color(reg["anomaly_type"])

            draw.polygon(pts, fill=cor)

            if DESENHAR_CONTORNO:
                draw.line(
                    pts + [pts[0]],
                    fill=cor,
                    width=ESPESSURA_CONTORNO
                )

        out_overlay = overlay_dir / f"{image_name}_overlay.png"
        out_img.save(out_overlay)

        comparison_img = make_side_by_side(
            original_img=original_img,
            overlay_img=out_img,
            gap=COMPARISON_GAP,
            gap_color=COR_GAP
        )

        out_comparison = comparison_dir / f"{image_name}_comparison.png"
        comparison_img.save(out_comparison)

        processadas += 1

    print(f"[OK] Processadas: {processadas}")
    print(f"[OK] Em falta:   {faltam_imagens}")

    return {
        "event_name": event_name,
        "excel_name": excel_path.name,
        "total_rows": total_rows,
        "total_valid": total_valid,
        "processed": processadas,
        "missing": faltam_imagens
    }


# ============================================================
# MAIN
# ============================================================

def main():
    pasta_excels = Path(PASTA_EXCELS_EVENTOS)
    pasta_imgs = Path(NOVA_PASTA_IMAGENS)
    pasta_saida = Path(PASTA_SAIDA)

    if not pasta_excels.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta_excels}")

    if not pasta_imgs.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta_imgs}")

    pasta_saida.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Pasta excels: {pasta_excels}")
    print(f"[INFO] Pasta imagens: {pasta_imgs}")
    print(f"[INFO] Pasta saída: {pasta_saida}")

    print("[INFO] A indexar imagens...")

    image_index = index_images_by_name(pasta_imgs)

    print(f"[INFO] Total imagens indexadas: {len(image_index)}")

    excel_files = sorted(pasta_excels.glob(EXCEL_GLOB))

    if not excel_files:
        raise FileNotFoundError("Nenhum Excel encontrado.")

    print(f"[INFO] Total excels: {len(excel_files)}")

    summary = []

    for excel_path in excel_files:
        result = process_one_event_excel(
            excel_path=excel_path,
            image_index=image_index,
            output_root=pasta_saida
        )
        summary.append(result)

    print("\n" + "=" * 70)
    print("PROCESSAMENTO CONCLUÍDO")
    print("=" * 70)

    total_processed = sum(x["processed"] for x in summary)
    total_missing = sum(x["missing"] for x in summary)

    print(f"Eventos: {len(summary)}")
    print(f"Imagens processadas: {total_processed}")
    print(f"Imagens em falta: {total_missing}")
    print(f"Saída: {pasta_saida}")


if __name__ == "__main__":
    main()

[INFO] Pasta excels: excel_result\degradacao_temporal_excels
[INFO] Pasta imagens: image
[INFO] Pasta saída: previews_degradacao_temporal
[INFO] A indexar imagens...
[INFO] Total imagens indexadas: 831
[INFO] Total excels: 20

[EVENTO] metricas_anomalias_faces_malha_degradacao_t01_00a_06m.xlsx
[INFO] Linhas totais: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Processadas: 387
[OK] Em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t02_01a_00m.xlsx
[INFO] Linhas totais: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Processadas: 387
[OK] Em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t03_01a_06m.xlsx
[INFO] Linhas totais: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Processadas: 387
[OK] Em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t04_02a_00m.xlsx
[INFO] Linhas totais: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Pr

KeyboardInterrupt: 